In [50]:
import cv2
import tensorflow as tf
import numpy as np
import os # For interacting with the file system
import random # For random colors in bounding boxes (used in helper function)

In [56]:
# Your base directory where models and images are located
base_directory = r"E:\SATHVIK\study\Projects\Object Detection Using Tensorflow"

# Path to your Faster R-CNN object detection model
# Make sure this directory contains the 'saved_model.pb' file and 'variables' folder
model_path = os.path.join(base_directory, "faster-rcnn-inception-resnet-v2-tensorflow2-1024x1024-v1")

# Path to your image file.
# Based on the uploaded file, your image is named 'image.jpeg'.
image_file_name = "image.jpeg" # <--- THIS IS THE CRUCIAL CORRECTION
image_path = os.path.join(base_directory, image_file_name)

# Load COCO labels from the internet (this is a standard file)
labels_path = tf.keras.utils.get_file(
    "coco_labels.txt",
    "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/data/mscoco_label_map.pbtxt"
)

# Read the labels into a Python list
with open(labels_path, "r") as f:
    labels = [line.strip() for line in f.readlines()]

print(f"Model path: {model_path}")
print(f"Image path: {image_path}")
print(f"Labels loaded: {len(labels)} classes")

Model path: E:\SATHVIK\study\Projects\Object Detection Using Tensorflow\faster-rcnn-inception-resnet-v2-tensorflow2-1024x1024-v1
Image path: E:\SATHVIK\study\Projects\Object Detection Using Tensorflow\image.jpeg
Labels loaded: 400 classes


In [57]:
try:
    # Load the TensorFlow SavedModel
    model = tf.saved_model.load(model_path)
    # Get the default serving signature for inference
    infer = model.signatures["serving_default"]
    print("Object detection model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    # Exit if the model cannot be loaded
    exit()

Object detection model loaded successfully!


In [58]:
def draw_box(image, ymin, xmin, ymax, xmax, class_name, confidence_score, color):
    """
    Draws a single bounding box and label on the image.
    """
    h, w, _ = image.shape
    left = int(xmin * w)
    top = int(ymin * h)
    right = int(xmax * w)
    bottom = int(ymax * h)

    cv2.rectangle(image, (left, top), (right, bottom), color, 2)

    text = f'{class_name}: {confidence_score:.2f}'
    font_scale = 0.5
    thickness = 2

    # Get text size for background rectangle
    (text_width, text_height) = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)[0]
    # Position the background rectangle slightly above the box
    text_offset_y = max(10, top - text_height - 10)
    cv2.rectangle(image, (left, text_offset_y - text_height - 5), (left + text_width + 2, text_offset_y), color, -1) # -1 for filled rectangle
    cv2.putText(image, text, (left + 2, text_offset_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, lineType=cv2.LINE_AA)


def draw_detections(image, boxes, classes, scores, class_names, threshold=0.5):
    """
    Loops through detections, filters by threshold, and draws bounding boxes.
    """
    color_codes = {} # To keep consistent colors for each class

    # Convert labels list to a NumPy array for easy indexing
    np_class_names = np.array(class_names)

    # Filter detections based on the confidence threshold
    valid_indices = np.where(scores >= threshold)
    filtered_boxes = boxes[valid_indices]
    filtered_classes = classes[valid_indices]
    filtered_scores = scores[valid_indices]

    for i in range(len(filtered_boxes)):
        box = filtered_boxes[i]
        ymin, xmin, ymax, xmax = box
        class_id = int(filtered_classes[i])
        score = filtered_scores[i]

        # Get the actual class name string
        class_name = np_class_names[class_id]
        if isinstance(class_name, bytes): # Decode if it's a byte string
            class_name = class_name.decode('utf-8')

        # Assign a random color if this class hasn't been seen yet
        if class_name not in color_codes:
            color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
            color_codes[class_name] = color
        else:
            color = color_codes[class_name]

        draw_box(image, ymin, xmin, ymax, xmax, class_name, score, color)
    return image

In [59]:
# This cell handles loading the image, running the model, and extracting detection info.

try:
    # Read the image using OpenCV
    frame = cv2.imread(image_path)

    if frame is None:
        print(f"Error: Could not read image at {image_path}. "
              "Please ensure the file exists and is a valid image format. "
              "If it has an extension (e.g., .jpg, .png), ensure it's included in the 'image_file_name' variable in Step 2.")
    else:
        print(f"Image '{image_file_name}' loaded successfully.")

        # Convert image from BGR (OpenCV default) to RGB (TensorFlow model expectation)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Convert the NumPy array image to a TensorFlow tensor
        input_tensor = tf.convert_to_tensor(rgb_frame)
        # Add an extra dimension to represent the batch (model expects [batch_size, height, width, channels])
        input_tensor = input_tensor[tf.newaxis, ...]

        # Run object detection
        detections = infer(input_tensor)

        print("Object detection performed.")
        # print("Raw detections:", {k: v.numpy() for k, v in detections.items()}) # Uncomment to see raw output

        # Extract detection information
        boxes = detections['detection_boxes'][0].numpy()
        classes = detections['detection_classes'][0].numpy().astype(np.int32)
        scores = detections['detection_scores'][0].numpy()
        num_detections = int(detections.pop('num_detections'))

        # Get original image dimensions for scaling bounding box coordinates
        im_height, im_width, _ = frame.shape

        # Create a copy of the original frame to draw on
        image_with_boxes = frame.copy()

        # Draw detections on the image
        image_with_boxes = draw_detections(image_with_boxes, boxes, classes, scores, labels, threshold=0.5)

        # Display the result
        cv2.imshow(f"Object Detection - {image_file_name}", image_with_boxes)

        # Wait indefinitely for a key press (0 means wait forever)
        # Press any key to close the window
        cv2.waitKey(0)

        # Destroy all OpenCV windows
        cv2.destroyAllWindows()

except Exception as e:
    print(f"An error occurred during processing: {e}")

Image 'image.jpeg' loaded successfully.
Object detection performed.
